In [3]:
!pip install google-genai

In [4]:
import os
import textwrap
from google import genai
from google.genai import errors
from google.colab import userdata

# Pull the secret directly from Colab's vault
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

# HR Policy Q&A Assistant: Prototype Demo & Evaluation

This notebook demonstrates the intelligent features of the HR RAG Assistant, including robust error handling, successful evaluation examples, and known failure edge cases.

In [5]:
import os
import textwrap
from google import genai
from google.genai import errors

# 1. Error Handling: API Key Validation
if "GEMINI_API_KEY" not in os.environ:
    print("CRITICAL ERROR: GEMINI_API_KEY environment variable is missing. Halting execution.")
else:
    client = genai.Client()

MOCK_HR_KNOWLEDGE_BASE = {
    "leave_policy": "Employees accrue 1.5 days of Paid Time Off (PTO) per month. Maximum rollover into the next calendar year is 5 days.",
    "remote_work": "Remote work is allowed up to 2 days per week for standard employees. Core hours are 10 AM to 3 PM EST."
}

def retrieve_context(query: str) -> str:
    query_lower = query.lower()
    if "leave" in query_lower or "pto" in query_lower:
        return MOCK_HR_KNOWLEDGE_BASE["leave_policy"]
    elif "remote" in query_lower or "home" in query_lower:
        return MOCK_HR_KNOWLEDGE_BASE["remote_work"]
    return ""

def ask_hr_assistant(query: str) -> str:
    context = retrieve_context(query)

    if not context:
        return "I cannot answer this based on the provided HR policies."

    prompt = f"""
    You are a strict HR Policy Assistant. Answer the employee's question using ONLY the provided context.
    If the context does not completely answer the question, state that clearly. Do not make up information.

    Context: {context}
    Question: {query}
    """

    # 2. Error Handling: API Call Try/Catch block
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
        )
        return response.text
    except errors.APIError as e:
        return f"API Connection Error: {e}"
    except Exception as e:
        return f"An unexpected system error occurred: {e}"

def chat_interface():
    """Simple notebook interface loop."""
    print("HR Assistant initialized. Type 'exit' to quit.")
    while True:
        user_input = input("\nEmployee: ")
        if user_input.lower() == 'exit':
            break
        print(f"Assistant: {ask_hr_assistant(user_input)}")

## Evaluation Examples (Success Cases)
*   **Prompt:** "What are the core hours for remote workers?"
*   **Expected Model Behavior:** Identifies the remote work context, extracts 10 AM to 3 PM EST, and formats a clear response without adding external assumptions.
*   **Prompt:** "Can I expense a $500 flight?"
*   **Expected Model Behavior:** Evaluates the context, finds no travel expense policy, and triggers the fallback refusal: "I cannot answer this based on the provided HR policies."

## Known Failure Cases
*   **Ambiguous Retrieval Failure:** If a user asks "Can I work from a coffee shop?", the mock retrieval looks for "remote" or "home". It fails to trigger the `remote_work` context, resulting in a false negative refusal. *Fix: Implement semantic vector search instead of keyword matching.*
*   **Context Nuance Failure:** If a user asks "I started on December 15th, how much PTO do I have on January 1st?", the context states "1.5 days per month", but doesn't define partial-month accrual. The LLM might attempt to do the math and hallucinate a pro-rated policy that doesn't exist.

In [6]:
# Execute the simple interface
chat_interface()

HR Assistant initialized. Type 'exit' to quit.

Employee: What are the core hours for remote workers?
Assistant: The core hours are 10 AM to 3 PM EST.

Employee: Can I expense a $500 flight?
Assistant: I cannot answer this based on the provided HR policies.

Employee: exit
